# Regime-Dependent Risk & Performance Attribution

**Research question:** Do full-sample risk metrics misrepresent asset risk profiles across market regimes?

**Thesis:** Yes. CAPM beta, volatility, VaR, and Sharpe ratio all vary materially across regimes. Full-sample averages are statistical artefacts that mask the true risk profile of every asset examined.

| | |
|---|---|
| **Assets** | NVDA, JPM, PG (equity); SPY (benchmark) |
| **Period** | 2019–2024 (1 000 trading days) |
| **Classifier** | SPY 30-day rolling vol split at in-sample median → Low-Vol / High-Vol |
| **Pipeline** | Regime detection → per-regime CAPM (HC3) → risk metrics → GARCH VaR → stress testing |

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Make src/ importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.config import TICKERS, RF_ANNUAL, setup_style
from src import data, regimes, metrics, capm, volatility, backtest, stress, plots

setup_style()

---
## 2. Data Loading

In [ ]:
prices = data.load_prices()
print(f"Loaded {len(prices)} trading days: "
      f"{prices.index[0].date()} → {prices.index[-1].date()}")
prices.describe().round(2)

---
## 3. Log Returns & Distributional Diagnostics

Log returns are time-additive and symmetric, making them suitable for compounding.
We confirm fat tails before any risk modelling — Gaussian VaR will systematically
underestimate tail risk if this assumption is violated.

In [ ]:
rets = data.log_returns(prices)
diag = data.return_diagnostics(rets)

print(f"{'Ticker':<6} {'Excess Kurt':>12} {'Skew':>8} {'JB p-val':>12} {'ADF p-val':>12}")
print('-' * 52)
for t, row in diag.iterrows():
    # bracket access: row.kurt and row.skew would resolve to Series methods
    print(f"{t:<6} {row['kurt']:>12.3f} {row['skew']:>8.3f} {row['jb_p']:>12.2e} {row['adf_p']:>12.2e}")

print("\nAll JB p-values ≈ 0 → normality rejected.  All ADF p-values ≈ 0 → returns stationary.")

In [ ]:
fig = plots.plot_qq(rets, diag)
fig.savefig('../data/qq_plots.png', dpi=150, bbox_inches='tight')

---
## 4. Regime Classification

SPY 30-day rolling volatility vs its full-sample median → binary Low-Vol / High-Vol labels.
Median split ensures balanced (~50/50) classes for reliable per-regime estimation.

> **Look-ahead note:** the full-sample median is acceptable for retrospective attribution.
> A live system must use an expanding-window median to avoid forward-looking bias.

In [ ]:
regime, rets_a, threshold = regimes.classify_regimes(rets)

vc = regime.value_counts()
for r, n in vc.items():
    print(f"  {r:<10}: {n:>4} days ({100*n/vc.sum():.1f}%)")
print(f"  Threshold  : {threshold:.5f} (daily σ)")

In [ ]:
spy_rvol = rets['SPY'].rolling(30).std()
fig = plots.plot_regimes(prices, regime, spy_rvol, threshold)
fig.savefig('../data/regime_chart.png', dpi=150, bbox_inches='tight')

---
## 5. Modelling
### 5a. Regime-Conditional Risk Metrics

In [ ]:
rows = []
for t in TICKERS:
    for lbl in ['Low-Vol', 'High-Vol', 'Full Sample']:
        r = rets_a[t] if lbl == 'Full Sample' else rets_a.loc[regime == lbl, t]
        m = metrics.calc_metrics(r, lbl)
        m['Ticker'] = t
        rows.append(m)

metrics_df = pd.DataFrame(rows)
cols = ['Label', 'N', 'AnnRet', 'AnnVol', 'Sharpe', 'Sortino', 'MaxDD', 'Calmar', 'VaR95', 'CVaR95']

for t in TICKERS:
    sub = metrics_df[metrics_df.Ticker == t][cols]
    print(f'\n{"="*90}\n  {t}  (RF={RF_ANNUAL*100:.1f}%)\n{"="*90}')
    print(sub.to_string(index=False, float_format=lambda x: f'{x:8.3f}'))

    lv = sub[sub.Label == 'Low-Vol'].iloc[0]
    hv = sub[sub.Label == 'High-Vol'].iloc[0]
    fs = sub[sub.Label == 'Full Sample'].iloc[0]
    print(f"  Sharpe gap (Low−High): {lv.Sharpe - hv.Sharpe:+.3f} | "
          f"CVaR stress mult: {hv.CVaR95/lv.CVaR95:.2f}x | "
          f"Full-sample Sharpe {fs.Sharpe:.3f} masks Low={lv.Sharpe:.3f}/High={hv.Sharpe:.3f}")

### 5b. Regime-Conditional CAPM — Beta Instability

HC3 robust standard errors are mandatory: fat-tailed, regime-switching returns
guarantee non-constant error variance, so plain OLS SEs are badly understated.
Δβ > 0 → assets become **more** market-exposed during stress — precisely when it hurts most.

In [ ]:
beta_df = capm.compute_regime_betas(rets_a, regime)

print(f"{'Ticker':<6}  {'Full β':<9} {'Low-Vol β':<11} {'High-Vol β':<12} {'Δβ':<8} {'Δβ%':<8} {'Full R²'}")
print('=' * 68)
for _, row in beta_df.iterrows():
    print(f"{row.Ticker:<6}  {row.Full_beta:>7.3f}   {row.LowVol_beta:>9.3f}   "
          f"{row.HighVol_beta:>10.3f}   {row.dBeta:>6.3f}   {row.dBeta_pct:>5.1f}%  {row.Full_r2:>6.3f}")
    print(f"        95% CI β (full): [{row.Full_blo:.3f}, {row.Full_bhi:.3f}]  p={row.Full_pb:.4f}")

In [ ]:
fig = plots.plot_beta_scatter(rets_a, regime, beta_df)
fig.savefig('../data/beta_scatter.png', dpi=150, bbox_inches='tight')

### 5c. Volatility Modelling — EWMA & GARCH(1,1)

Best GARCH error distribution selected by AIC across {normal, t, skewed-t}.
Returns scaled to % internally for GARCH numerical stability.

In [ ]:
vol_models = volatility.fit_all_vol_models(rets_a)

for t, m in vol_models.items():
    p = m['garch_fit'].params if m['garch_fit'] else {}
    persist = p.get('alpha[1]', 0) + p.get('beta[1]', 0)
    print(f"{t}: GARCH dist={m['garch_dist']}, "
          f"α+β={persist:.4f}{'  ⚠ IGARCH' if persist >= 1 else ''}  | "
          f"EWMA fcast={m['ewma_fcast']*np.sqrt(252)*100:.2f}%  "
          f"GARCH fcast={m['garch_fcast']*np.sqrt(252)*100:.2f}%")

In [ ]:
fig = plots.plot_volatility(vol_models)
fig.savefig('../data/volatility.png', dpi=150, bbox_inches='tight')

---
## 6. Evaluation
### 6a. VaR Backtesting — Kupiec POF + Christoffersen Independence

A well-calibrated VaR model must pass **both** tests:
- **POF** (Kupiec 1995): violation frequency ≈ 5%.
- **IND** (Christoffersen 1998): violations not clustered.

`PASS` = fail to reject H₀ at 5% → correct model behaviour.

In [ ]:
bt_df = backtest.backtest_var(rets_a, regime)

print(f"{'Ticker':<6} {'Regime':<12} {'n':<6} {'Exceed':<7} {'Rate%':<7} "
      f"{'POF p':<10} {'POF':<6} {'IND p':<10} {'IND'}")
print('=' * 78)
for _, row in bt_df.iterrows():
    # bracket access: row.T would be the Series transpose, not the column
    pof = 'PASS' if row['pof_pass'] else 'FAIL'
    ind = 'PASS' if row['ind_pass'] else 'FAIL'
    print(f"{row['Ticker']:<6} {row['Regime']:<12} {row['T']:<6} {row['N']:<7} "
          f"{row['pi_hat']*100:<7.2f} {row['p_pof']:<10.4f} {pof:<6} {row['p_ind']:<10.4f} {ind}")

### 6b. Historical Stress Testing

Equal-weight portfolio (25% each) evaluated during two historically identified stress events.
Weights are parameterised — change `portfolio_weights` to test any allocation.

In [ ]:
portfolio_weights = {'NVDA': 0.25, 'JPM': 0.25, 'PG': 0.25, 'SPY': 0.25}

asset_records, portfolio_summary = stress.run_stress_tests(
    rets_a, weights=portfolio_weights
)

asset_df = pd.DataFrame(asset_records)
for name, grp in asset_df.groupby('event'):
    print(f'\n{"="*65}\n  {name}\n{"="*65}')
    print(grp[['ticker','n_days','cum_ret_pct','max_dd_pct','stress_beta']]
          .to_string(index=False, float_format=lambda x: f'{x:.3f}'))

print('\nPortfolio summary:')
for s in portfolio_summary:
    print(f"  {s['name']}: cum={s['p_cum']*100:+.2f}%  "
          f"P&L=${s['dollar_pnl']:+,.0f}  "
          f"daily CVaR={s['p_cvar']*100:.3f}%")

In [ ]:
# ── Save the numbers behind the README Key Results block ───────────────────
import json
RESULTS_DIR = Path.cwd().parent / 'results'; RESULTS_DIR.mkdir(exist_ok=True)
m = metrics_df.set_index(['Ticker', 'Label'])
out = {
    'sample': {'start': str(prices.index[0].date()), 'end': str(prices.index[-1].date()),
               'n_days': int(len(rets_a)), 'high_vol_share': float((regime == 'High-Vol').mean())},
    'assets': {},
    'portfolio_stress': {s['name']: float(s['p_cum']) for s in portfolio_summary},
}
for t in TICKERS:
    a = {
        'sharpe': {lbl: float(m.loc[(t, lbl), 'Sharpe']) for lbl in ['Low-Vol', 'High-Vol', 'Full Sample']},
        'cvar_mult': float(m.loc[(t, 'High-Vol'), 'CVaR95'] / m.loc[(t, 'Low-Vol'), 'CVaR95']),
        'garch_dist': vol_models[t]['garch_dist'],
        'garch_persistence': float(vol_models[t]['garch_fit'].params.get('alpha[1]', 0)
                                   + vol_models[t]['garch_fit'].params.get('beta[1]', 0)),
    }
    b = beta_df.set_index('Ticker')
    if t in b.index:
        a['beta'] = {'low': float(b.loc[t, 'LowVol_beta']), 'high': float(b.loc[t, 'HighVol_beta']),
                     'dbeta_pct': float(b.loc[t, 'dBeta_pct'])}
    bt = bt_df[(bt_df.Ticker == t) & (bt_df.Regime == 'Full Sample')].iloc[0]
    a['var_backtest_full'] = {'exceed_rate': float(bt['pi_hat']), 'p_pof': float(bt['p_pof']),
                              'p_ind': float(bt['p_ind'])}
    out['assets'][t] = a
(RESULTS_DIR / 'key_results.json').write_text(json.dumps(out, indent=2) + '\n')
print(json.dumps(out['portfolio_stress'], indent=2))

---
## 7. Summary Dashboard

In [ ]:
fig = plots.plot_dashboard(
    metrics_df=metrics_df,
    regime=regime,
    beta_df=beta_df,
    vol_models=vol_models,
    stress_summary=portfolio_summary,
    save_path='../data/regime_dashboard.png',
)
print('Dashboard saved → data/regime_dashboard.png')